# Queue-Based Market Making in Large Tick Size Assets

## Overview
The significance of queue position is well-known in microstructure trading, particularly in assets with large tick sizes. This is because larger tick assets typically more constrained price movements. The impact of tick size is discussed in detail in ["Large tick assets: implicit spread and optimal tick size"](https://arxiv.org/pdf/1207.6325).

![CRVUSDT_chart](https://github.com/nkaz001/hftbacktest/blob/master/docs/images/CRVUSDT_chart.png)

<div class="alert alert-info">
    
**Note:** This example is for educational purposes only and demonstrates effective strategies for high-frequency market-making schemes. All backtests are based on a 0.005% rebate, the highest market maker rebate available on Binance Futures. See <a href="https://www.binance.com/en/support/announcement/binance-updates-usd%E2%93%A2-margined-futures-liquidity-provider-program-2024-06-03-fefc6aa25e0947e2bf745c1c56bea13e">Binance Upgrades USDⓢ-Margined Futures Liquidity Provider Program</a> for more details.
    
</div>

## Book Pressure
To begin, we will review the [Market Microstructure signals described in this article](https://blog.headlandstech.com/2017/08/), which are similar to the concept of micro-price. Book imbalance is also addressed in [Market Making with Alpha - Order Book Imbalance](https://github.com/nkaz001/hftbacktest/blob/master/examples/Market%20Making%20with%20Alpha%20-%20Order%20Book%20Imbalance.ipynb).

## Runnable Tardis Test

This notebook executes the corresponding experiment through the shared
`tutorial_reproduction` runner. It uses existing Tardis files only and
does not require a Tardis API key or download data.

Defaults:

- amdserver: `/home/molly/data/tardis/binance-futures`, `2025-08-01`
- Mac: `~/Documents/tardis`, `2025-01-01`
- Window: `300` seconds

Optional environment overrides:

- `HFTBACKTEST_TARDIS_ROOT`
- `HFTBACKTEST_MULTI_TARDIS_ROOT`
- `HFTBACKTEST_TARDIS_DATE`
- `HFTBACKTEST_NOTEBOOK_SECONDS`
- `HFTBACKTEST_NOTEBOOK_OUTPUT`

Active experiment: `Queue-Based Market Making in Large Tick Size Assets.ipynb` (`queue_based_market_making_large_tick`).
This file is a generated copy; the original notebook under `examples/` remains unchanged.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
search_roots = []
for base in (cwd, *cwd.parents):
    search_roots.extend((base, base / 'examples'))
examples_root = next(
    path for path in search_roots
    if (path / 'tutorial_reproduction').is_dir()
)
if str(examples_root) not in sys.path:
    sys.path.insert(0, str(examples_root))

from tutorial_reproduction.notebook_support import (
    context_dict,
    notebook_context,
    run_notebook_experiment,
)

In [ ]:
context = notebook_context('0804T005')
context_dict(context)

In [ ]:
manifest = run_notebook_experiment('queue_based_market_making_large_tick', context)
manifest['result']

In [ ]:
assert manifest['result']['status'] != 'failed'
print('notebook:', manifest['notebook'])
print('status:', manifest['result']['status'])
print('output:', context.output_root)

## Original Tutorial Reference

The original tutorial narrative and code are retained below for comparison.
Original code cells are rendered as non-executing references so that
`Run All` remains reproducible with the configured Tardis dataset.

## Trade Impulse

Let's examine how it changes when we incorporate the trade impulse.

There is not much difference, as the last trade quantity is relatively small compared to the best bid and offer quantities.

![CRVUSDT_depth](https://github.com/nkaz001/hftbacktest/blob/master/docs/images/CRVUSDT_depth.png)

The following example demonstrates a variant of trade impulse using aggregated trade quantities.

You can also adjust `trade_impulse_adj` to modify the impact of the trade impulse. Alternatively, you can explore other ways to compute the trade impulse, such as `(best_bid * best_ask_qty + best_ask * best_bid_qty + last_px * last_qty) / (best_bid_qty + best_ask_qty + last_qty)`, VWAP, etc.

## Pure Queue-Based Model

One possible reason for this strategy's profitability is the limited price movement due to the large tick size. For instance, CRVUSDT has a tick size of 38 basis points (0.001 / 0.26 * 10,000), which is comparatively very larger than BTCUSDT, where the tick size is approximately 0.018 basis points (0.1 / 54,000 * 10,000).

This also highlights the importance of queue position modeling in fill simulations for assets with large tick sizes.

In the CRVUSDT charts shown above, observing the trading activities, you can see that most trades occur at the best bid and ask prices, with little change in the overall price level.

This suggests an opportunity to adjust the microstructure signal into a purely queue-based signal. For example, if there is sufficient quantity to maintain the price level, preventing it from moving adversely, we can choose to maintain our quote. Let’s explore how this can be implemented in a simplified form.

You can also explore more sophisticated approaches, such as dynamically controlling the `qty_threshold` and integrating it with the skew value, for example, `qty_threshold * (1 ± skew_val)`, similar to how skew is applied to the price. In other words, in the previous example, the spread is set in terms of price, but you can set the spread in terms of queue such as the queue position, the queue behind the order, the total queue, etc.

Additionally, instead of reacting at fixed intervals, it may be more effective to respond to each incoming feed. This allows for faster reactions when the quantity at the BBO decreases rapidly, helping to avoid adverse selection. You can test this approach using the `wait_next_feed` method.